In [6]:
import tkinter as tk
from tkinter import messagebox, ttk
import uuid

In [7]:


# Student class
class Student:
    def __init__(self, name, age, grade):
        self.id = str(uuid.uuid4())  # Unique ID
        self.name = name
        self.age = age
        self.grade = grade

    def __str__(self):
        return f"{self.name} (Age: {self.age}, Grade: {self.grade})"

# Manager class
class StudentManager:
    def __init__(self):
        self.students = {}

    def add_student(self, student):
        self.students[student.id] = student

    def update_student(self, student_id, name, age, grade):
        if student_id in self.students:
            self.students[student_id].name = name
            self.students[student_id].age = age
            self.students[student_id].grade = grade

    def delete_student(self, student_id):
        if student_id in self.students:
            del self.students[student_id]

    def search_students(self, keyword):
        return [student for student in self.students.values() if keyword.lower() in student.name.lower()]

    def get_all_students(self):
        return list(self.students.values())

# GUI Class
class StudentApp:
    def __init__(self, root):
        self.manager = StudentManager()
        self.root = root
        self.root.title("Advanced Student Management")
        self.root.geometry("700x500")
        self.selected_student_id = None
        self.build_ui()

    def build_ui(self):
        input_frame = ttk.Frame(self.root, padding=10)
        input_frame.pack(fill='x')

        # Name
        ttk.Label(input_frame, text="Name:").grid(row=0, column=0, padx=5, pady=5)
        self.name_entry = ttk.Entry(input_frame)
        self.name_entry.grid(row=0, column=1, padx=5, pady=5)

        # Age
        ttk.Label(input_frame, text="Age:").grid(row=1, column=0, padx=5, pady=5)
        self.age_entry = ttk.Entry(input_frame)
        self.age_entry.grid(row=1, column=1, padx=5, pady=5)

        # Grade
        ttk.Label(input_frame, text="Grade:").grid(row=2, column=0, padx=5, pady=5)
        self.grade_entry = ttk.Entry(input_frame)
        self.grade_entry.grid(row=2, column=1, padx=5, pady=5)

        # Buttons
        btn_frame = ttk.Frame(self.root, padding=10)
        btn_frame.pack(fill='x')

        ttk.Button(btn_frame, text="Add", command=self.add_student).pack(side='left', padx=5)
        ttk.Button(btn_frame, text="Update", command=self.update_student).pack(side='left', padx=5)
        ttk.Button(btn_frame, text="Delete", command=self.delete_student).pack(side='left', padx=5)

        # Search
        search_frame = ttk.Frame(self.root, padding=10)
        search_frame.pack(fill='x')

        self.search_entry = ttk.Entry(search_frame)
        self.search_entry.pack(side='left', padx=5)
        ttk.Button(search_frame, text="Search", command=self.search_students).pack(side='left', padx=5)
        ttk.Button(search_frame, text="Show All", command=self.refresh_table).pack(side='left', padx=5)

        # Table
        self.table = ttk.Treeview(self.root, columns=("ID", "Name", "Age", "Grade"), show="headings")
        for col in ("ID", "Name", "Age", "Grade"):
            self.table.heading(col, text=col)
            self.table.column(col, width=150)
        self.table.pack(fill='both', expand=True, padx=10, pady=10)

        self.table.bind('<<TreeviewSelect>>', self.select_student)

    def clear_entries(self):
        self.name_entry.delete(0, tk.END)
        self.age_entry.delete(0, tk.END)
        self.grade_entry.delete(0, tk.END)
        self.selected_student_id = None

    def validate_inputs(self):
        name = self.name_entry.get().strip()
        age_text = self.age_entry.get().strip()
        grade = self.grade_entry.get().strip()
        if not (name and age_text and grade):
            messagebox.showwarning("Input Error", "All fields are required!")
            return None
        try:
            age = int(age_text)
        except ValueError:
            messagebox.showerror("Input Error", "Age must be a number.")
            return None
        return name, age, grade

    def add_student(self):
        validated = self.validate_inputs()
        if validated:
            name, age, grade = validated
            student = Student(name, age, grade)
            self.manager.add_student(student)
            self.refresh_table()
            self.clear_entries()

    def update_student(self):
        if not self.selected_student_id:
            messagebox.showwarning("Selection Error", "No student selected!")
            return
        validated = self.validate_inputs()
        if validated:
            name, age, grade = validated
            self.manager.update_student(self.selected_student_id, name, age, grade)
            self.refresh_table()
            self.clear_entries()

    def delete_student(self):
        if not self.selected_student_id:
            messagebox.showwarning("Selection Error", "No student selected!")
            return
        self.manager.delete_student(self.selected_student_id)
        self.refresh_table()
        self.clear_entries()

    def search_students(self):
        keyword = self.search_entry.get().strip()
        results = self.manager.search_students(keyword)
        self.refresh_table(results)

    def refresh_table(self, students=None):
        for item in self.table.get_children():
            self.table.delete(item)
        if students is None:
            students = self.manager.get_all_students()
        for student in students:
            self.table.insert('', 'end', iid=student.id, values=(student.id, student.name, student.age, student.grade))

    def select_student(self, event):
        selected = self.table.selection()
        if selected:
            student_id = selected[0]
            student = self.manager.students.get(student_id)
            if student:
                self.selected_student_id = student.id
                self.name_entry.delete(0, tk.END)
                self.name_entry.insert(0, student.name)
                self.age_entry.delete(0, tk.END)
                self.age_entry.insert(0, student.age)
                self.grade_entry.delete(0, tk.END)
                self.grade_entry.insert(0, student.grade)

if __name__ == "__main__":
    root = tk.Tk()
    app = StudentApp(root)
    root.mainloop()
